# Experiment B reproduction

This notebook reproduces the final Experiment B design-geometry quantities and paper figure from the released processed inputs and final numerical results. By default it does **not** download or rebuild the raw Intel/NDBC datasets, and it never overwrites files in `data/` or `figures/`.

The primary quantities are $N\_{\mathrm{eff}}(\pi)=1/\sum\_i\pi\_i^2$ and $G(\pi)=[L(\pi)-L(U)]/[L(D)-L(U)]$, with $L(\pi)=\log\det(\sum\_i\pi\_iC\_i)$.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg as la
from scipy.optimize import minimize

ROOT = Path.cwd().resolve()
DATA = ROOT / "data"
FIGURES = ROOT / "figures"
REPRODUCED = ROOT / "reproduced"
REPRODUCED.mkdir(exist_ok=True)

P_GRID = np.array([2.2, 2.5, 3, 4, 6, 8, 12, 16, 24, 32, 49, 64.0])

print("Package root:", ROOT)
print("Data directory:", DATA)


## 1. Load the released inputs and final numerical results

In [ ]:
intel_input = np.load(DATA / "B_intel_processed_input_frozen.npz", allow_pickle=False)
gulf_input = np.load(DATA / "B_gulf_processed_input_frozen.npz", allow_pickle=False)
intel_raw = np.load(DATA / "B_final_intel_raw.npz", allow_pickle=False)
gulf_raw = np.load(DATA / "B_final_gulf_raw.npz", allow_pickle=False)
summary = pd.read_csv(DATA / "B_final_summary.csv")
baselines = pd.read_csv(DATA / "B_final_baselines.csv")

print("Intel blocks:", intel_input["B_blocks"].shape, "C:", intel_input["C_blocks"].shape)
print("Gulf blocks :", gulf_input["A_blocks"].shape, "C:", gulf_input["C_blocks"].shape)
print("Final summary rows:", len(summary))

np.testing.assert_allclose(intel_raw["p_grid"], P_GRID, rtol=0, atol=0)
np.testing.assert_allclose(gulf_raw["p_grid"], P_GRID, rtol=0, atol=0)
assert len(summary) == 24


## 2. Check the released block geometry

Intel uses $C\_i=B\_i^\top B\_i$. Gulf uses the whitened factors $A\_i$ and $C\_i=A\_i^\top A\_i$.

In [ ]:
intel_B = intel_input["B_blocks"]
intel_C = intel_input["C_blocks"]
gulf_A = gulf_input["A_blocks"]
gulf_C = gulf_input["C_blocks"]

intel_C_rebuilt = np.einsum("kri,krj->kij", intel_B, intel_B)
gulf_C_rebuilt = np.einsum("kri,krj->kij", gulf_A, gulf_A)

np.testing.assert_allclose(intel_C_rebuilt, intel_C, rtol=0, atol=5e-15)
np.testing.assert_allclose(gulf_C_rebuilt, gulf_C, rtol=0, atol=5e-12)
np.testing.assert_allclose(intel_C.sum(axis=0), np.eye(30), rtol=0, atol=2e-14)

print("Intel sensor IDs:", intel_input["sensor_ids"].tolist())
print("Gulf buoy IDs   :", gulf_input["buoy_ids"].tolist())
print("Intel ||sum C - I||_F:", np.linalg.norm(intel_C.sum(axis=0) - np.eye(30), ord="fro"))
print("Gulf aggregate eigenvalues:", np.linalg.eigvalsh(gulf_C.sum(axis=0)))


## 3. Recompute the plotted metrics from the released allocations

In [ ]:
def information_matrix(pi, C_blocks):
    M = np.tensordot(np.asarray(pi, float), np.asarray(C_blocks, float), axes=(0, 0))
    return 0.5 * (M + M.T)

def logdet_information(pi, C_blocks):
    M = information_matrix(pi, C_blocks)
    L = np.linalg.cholesky(M)
    return float(2.0 * np.log(np.diag(L)).sum())

def effective_support(pi):
    pi = np.asarray(pi, float)
    return float(1.0 / np.sum(pi * pi))

def normalized_gain(pi, C_blocks, U, D):
    L_U = logdet_information(U, C_blocks)
    L_D = logdet_information(D, C_blocks)
    L_pi = logdet_information(pi, C_blocks)
    return float((L_pi - L_U) / (L_D - L_U))

def reconstruct_dataset(name, raw, C_blocks):
    U = raw["uniform_allocation"]
    D = raw["D_direct"]
    rows = []
    for p, pi, w in zip(raw["p_grid"], raw["allocations"], raw["raw_weights"]):
        rho = 1.0 - 2.0 / float(p)
        pi_from_w = w ** rho
        pi_from_w = pi_from_w / pi_from_w.sum()
        np.testing.assert_allclose(pi_from_w, pi, rtol=0, atol=8e-17)
        rows.append({
            "dataset": name,
            "p": float(p),
            "effective_support": effective_support(pi),
            "logdet_information": logdet_information(pi, C_blocks),
            "G_uniform_to_D": normalized_gain(pi, C_blocks, U, D),
        })
    return pd.DataFrame(rows)

recomputed = pd.concat([
    reconstruct_dataset("Intel", intel_raw, intel_C),
    reconstruct_dataset("Gulf", gulf_raw, gulf_C),
], ignore_index=True)

merged = summary.merge(recomputed, on=["dataset", "p"], suffixes=("_saved", "_recomputed"))
for col in ["effective_support", "logdet_information", "G_uniform_to_D"]:
    np.testing.assert_allclose(
        merged[f"{col}_saved"], merged[f"{col}_recomputed"], rtol=0, atol=5e-13
    )

for dataset, raw, C in [("Intel", intel_raw, intel_C), ("Gulf", gulf_raw, gulf_C)]:
    base = baselines.loc[baselines["dataset"] == dataset].iloc[0]
    U, D = raw["uniform_allocation"], raw["D_direct"]
    np.testing.assert_allclose(effective_support(U), base["N_eff_uniform"], atol=1e-12)
    np.testing.assert_allclose(effective_support(D), base["N_eff_D"], atol=1e-12)
    np.testing.assert_allclose(logdet_information(U, C), base["L_uniform"], atol=1e-12)
    np.testing.assert_allclose(logdet_information(D, C), base["L_D"], atol=1e-12)

print("All released allocations and plotted metrics reconstruct successfully.")
display(recomputed)


## 4. Reproduce the paper figure

In [ ]:
LABEL_P = {2.2, 4.0, 8.0, 16.0, 32.0, 64.0}
P_LABEL_OFFSETS = {
    "Intel": {2.2:(6,5), 4.0:(5,5), 8.0:(5,5), 16.0:(5,5), 32.0:(7,-10), 64.0:(4,9)},
    "Gulf":  {2.2:(7,-2), 4.0:(6,5), 8.0:(6,5), 16.0:(6,5), 32.0:(6,5), 64.0:(6,7)},
}
ENDPOINT_OFFSETS = {
    "Intel": {"uniform":(-5,-23), "D-opt":(7,-11)},
    "Gulf":  {"uniform":(-4,-23), "D-opt":(7,-11)},
}

fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.5), sharey=True, constrained_layout=True)
for ax, dataset in zip(axes, ["Intel", "Gulf"]):
    sdf = summary[summary["dataset"] == dataset].sort_values("p").reset_index(drop=True)
    base = baselines[baselines["dataset"] == dataset].iloc[0]
    x_path = np.concatenate([[base["N_eff_uniform"]], sdf["effective_support"].to_numpy(), [base["N_eff_D"]]])
    y_path = np.concatenate([[0.0], sdf["G_uniform_to_D"].to_numpy(), [1.0]])
    ax.plot(x_path, y_path, color="lightsteelblue", linewidth=1.8, zorder=1)
    ax.scatter(sdf["effective_support"], sdf["G_uniform_to_D"], color="mediumvioletred", s=31, zorder=3)
    ax.scatter([base["N_eff_uniform"]], [0.0], marker="s", facecolors="white", edgecolors="mediumvioletred", linewidths=1.5, s=56, zorder=4)
    ax.scatter([base["N_eff_D"]], [1.0], marker="*", color="mediumvioletred", s=95, zorder=4)
    ax.annotate("uniform", (base["N_eff_uniform"], 0.0), xytext=ENDPOINT_OFFSETS[dataset]["uniform"], textcoords="offset points", ha="right", va="top", fontsize=8.5, annotation_clip=False)
    ax.annotate("D-opt", (base["N_eff_D"], 1.0), xytext=ENDPOINT_OFFSETS[dataset]["D-opt"], textcoords="offset points", ha="left", va="top", fontsize=8.5)
    for _, row in sdf.iterrows():
        p = float(row["p"])
        if p in LABEL_P:
            dx, dy = P_LABEL_OFFSETS[dataset][p]
            ax.annotate(rf"$p={p:g}$", (row["effective_support"], row["G_uniform_to_D"]), xytext=(dx,dy), textcoords="offset points", fontsize=7.5)
    ax.set_xlabel(r"Effective support $N_{\mathrm{eff}}(\pi)$")
    ax.set_ylim(-0.035, 1.045)
    ax.grid(alpha=0.18, linewidth=0.6)
    ax.text(0.97, 0.95, dataset, transform=ax.transAxes, ha="right", va="top", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel(r"Normalized log-det gain $G(\pi)$")

pdf_out = REPRODUCED / "B_design_geometry_reproduced.pdf"
png_out = REPRODUCED / "B_design_geometry_reproduced.png"
fig.savefig(pdf_out, format="pdf", bbox_inches="tight", pad_inches=0.03)
fig.savefig(png_out, dpi=180, bbox_inches="tight", pad_inches=0.03)
plt.show()
print("Wrote:", pdf_out.relative_to(ROOT))
print("Wrote:", png_out.relative_to(ROOT))


## 5. Optional full numerical rerun

The released figure can be reproduced without rerunning the fixed-point solver. For an additional check, set `RUN_FULL_RECOMPUTE = True` below. This reruns the finite-$p$ block-Lewis iterations from the frozen processed blocks and reruns the direct SLSQP D-optimal solve. It writes no replacement data.

In [ ]:
RUN_FULL_RECOMPUTE = False

def block_update_qr(A_blocks, w, p):
    A_blocks = np.asarray(A_blocks, dtype=float)
    w = np.asarray(w, dtype=float)
    k, block_rows, d = A_blocks.shape
    rho = 1.0 - 2.0 / float(p)
    A_stack = A_blocks.reshape(k * block_rows, d)
    row_scale = np.repeat(w ** (rho / 2.0), block_rows)
    B_stack = row_scale[:, None] * A_stack
    _, R = np.linalg.qr(B_stack, mode="reduced")
    X_stack = la.solve_triangular(R.T, A_stack.T, lower=True, check_finite=False).T
    X_blocks = X_stack.reshape(k, block_rows, d)
    return (w ** rho) * np.sum(X_blocks ** 2, axis=(1, 2))

def run_block_lewis(A_blocks, p, tol=1e-8, max_updates=10_000):
    A_blocks = np.asarray(A_blocks, dtype=float)
    k, _, d = A_blocks.shape
    ranks = np.asarray([np.linalg.matrix_rank(A_i) for A_i in A_blocks], dtype=float)
    w = d * ranks / ranks.sum()
    for update in range(1, max_updates + 1):
        w_next = block_update_qr(A_blocks, w, p)
        residual = float(np.max(np.abs(np.log(w_next / w))))
        w = w_next
        if residual <= tol:
            rho = 1.0 - 2.0 / float(p)
            pi = w ** rho
            pi = pi / pi.sum()
            return w, pi, update, residual
    raise RuntimeError(f"cap reached at p={p}")

def dopt_value_gradient(pi, C_blocks):
    M = information_matrix(pi, C_blocks)
    try:
        chol = np.linalg.cholesky(M)
    except np.linalg.LinAlgError:
        return None, None
    logdet = float(2.0 * np.log(np.diag(chol)).sum())
    M_inv = la.cho_solve((chol, True), np.eye(M.shape[0]), check_finite=False)
    sensitivities = np.einsum("kij,ji->k", C_blocks, M_inv)
    return logdet, sensitivities

def solve_dopt_direct(C_blocks):
    C_blocks = np.asarray(C_blocks, dtype=float)
    k = C_blocks.shape[0]
    x0 = np.full(k, 1.0 / k)
    def objective(pi):
        val, _ = dopt_value_gradient(pi, C_blocks)
        return 1e100 if val is None else -val
    def gradient(pi):
        val, sens = dopt_value_gradient(pi, C_blocks)
        return np.zeros(k) if val is None else -sens
    res = minimize(objective, x0, jac=gradient, method="SLSQP", bounds=[(0.0,1.0)]*k, constraints=[{"type":"eq", "fun":lambda pi: np.sum(pi)-1.0, "jac":lambda pi: np.ones_like(pi)}], options={"ftol":1e-12, "maxiter":10_000, "disp":False})
    if not res.success:
        raise RuntimeError(res.message)
    return np.asarray(res.x, float)

if RUN_FULL_RECOMPUTE:
    checks = []
    for name, A, C, raw in [("Intel", intel_B, intel_C, intel_raw), ("Gulf", gulf_A, gulf_C, gulf_raw)]:
        for j, p in enumerate(P_GRID):
            w, pi, updates, residual = run_block_lewis(A, p)
            checks.append((name, p, updates, float(np.max(np.abs(pi - raw["allocations"][j])))))
        D_new = solve_dopt_direct(C)
        print(name, "max D allocation difference:", np.max(np.abs(D_new - raw["D_direct"])))
    checks = pd.DataFrame(checks, columns=["dataset","p","updates","max_allocation_difference"])
    print(checks.to_string(index=False))
else:
    print("Full numerical rerun skipped. Set RUN_FULL_RECOMPUTE = True to enable it.")


## 6. Compact released-result summary

In [ ]:
compact = summary[summary["p"].isin([2.2, 16.0, 64.0])][["dataset","p","effective_support","G_uniform_to_D"]].copy()
print(compact.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\nD-optimal effective support:")
print(baselines[["dataset","N_eff_D"]].to_string(index=False, float_format=lambda x: f"{x:.6f}"))
